# 30_01 - Train CustomCNN v1 From Scratch

## Goal
Train the Phase 3 baseline scratch CNN (`CustomCNN v1`) on the fixed dataset split `split_v1` using the shared Phase 1 transformation pipeline.

## Why This Architecture?

CustomCNN v1 is built to be a **simple, honest starting point**. Before trying anything fancy (like using pretrained models), we want to know: *what can a plain CNN trained from scratch actually achieve on this dataset?* That's the only job of this model.

### The Structure

The network has two parts - a **feature extractor** and a **classifier**.

The feature extractor is 3 convolutional blocks, each doing the same thing: look for patterns in the image, then shrink the image in half. The first block looks for basic things like edges and colours. Each block after that builds on the previous one, picking up more complex patterns like textures and shapes. By the end, we have a compact summary of what's visually interesting in the image.

The classifier then takes that summary and makes a decision: is this a cat, a dog, or wildlife?

### Why These Specific Choices?

- **Doubling channels (32 → 64 → 128)** - a well-known pattern that gives the network more "vocabulary" to describe what it sees as images get more complex deeper in the network.
- **Global average pooling instead of flatten** - rather than dumping all remaining pixels into a giant vector, we take one average per channel. Keeps the model small and avoids unnecessary parameters.
- **One hidden layer (256 neurons)** - for a 3-class problem, this is plenty. More layers would just risk memorising the training data.
- **Dropout 0.5** - randomly switches off half the neurons during training, forcing the network to not rely too heavily on any single feature. Makes it generalise better.

### Why It Works Well Enough

The three classes - cats, dogs, wildlife - are visually quite distinct. You don't need a deep or complex model to tell them apart reliably. This model proves that with **~127K parameters** (tiny by modern standards), trained from zero, you can still reach **94.5% accuracy**. That's a useful result on its own, and it gives every future experiment a concrete bar to beat.

## Setup

**Cells 1–6** handle environment setup: imports, project-root detection, `sys.path` extension, seed initialisation for reproducibility (`SEED=42`), required-file pre-flight checks, and MLflow experiment configuration (`AnimalClassification` → local `mlruns/`).

> ⚠️ A benign `TqdmWarning` (ipywidgets) and an MLflow `FutureWarning` about the file-based tracking store may appear - both are cosmetic and do not affect execution.

In [ ]:
# Cell 1 - Imports

import os
import sys
import json
import random
from pathlib import Path

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import DataLoader

import mlflow

/home/temporaryuser4/Desktop/rufiz/~/AnimalClassification/AnimalClassification_server/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# Cell 2 - Project root + paths (preserve Phase 2 conventions)

NOTEBOOK_DIR = Path.cwd().expanduser().resolve()

def find_project_root(start: Path) -> Path:
    markers_all = ["src", "configs"]
    markers_any = [".git", "pyproject.toml", "requirements.txt"]

    for p in [start] + list(start.parents):
        has_all = all((p / m).exists() for m in markers_all)
        has_any = any((p / m).exists() for m in markers_any)
        if has_all and has_any:
            return p

    for p in [start] + list(start.parents):
        if all((p / m).exists() for m in markers_all):
            return p

    raise RuntimeError(f"Could not locate project root from: {start}")

PROJECT_ROOT = find_project_root(NOTEBOOK_DIR)

CONFIGS_DIR = PROJECT_ROOT / "configs"
DATA_DIR = PROJECT_ROOT / "data"
REPORTS_DIR = PROJECT_ROOT / "reports"
MODELS_DIR = PROJECT_ROOT / "models"
TRACKING_DIR = PROJECT_ROOT / "mlruns"

SPLIT_ID = "split_v1"
SPLITS_DIR = DATA_DIR / "splits" / SPLIT_ID
TRAIN_CSV = SPLITS_DIR / "train.csv"
VAL_CSV = SPLITS_DIR / "val.csv"
TEST_CSV = SPLITS_DIR / "test.csv"
CLASSES_JSON = SPLITS_DIR / "classes.json"

TRANSFORMS_CONFIG_PATH = CONFIGS_DIR / "transforms_v1.yaml"
TRANSFORM_ID_TRAIN = "transforms_v1_train_runtime_aug"
TRANSFORM_ID_EVAL = "transforms_v1_eval_resize256_centercrop224_imagenetnorm"

MODEL_NAME = "customcnn_v1"
MODEL_FAMILY_DIR = MODELS_DIR / "cnn_scratch" / MODEL_NAME
REPORTS_METRICS_DIR = REPORTS_DIR / "metrics"
REPORTS_FIGURES_DIR = REPORTS_DIR / "figures"

MODEL_FAMILY_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_METRICS_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_FIGURES_DIR.mkdir(parents=True, exist_ok=True)
TRACKING_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT     :", PROJECT_ROOT)
print("MODEL_FAMILY_DIR :", MODEL_FAMILY_DIR)
print("TRACKING_DIR     :", TRACKING_DIR)

PROJECT_ROOT     : /home/temporaryuser4/Desktop/rufiz/~/AnimalClassification/AnimalClassification_server
MODEL_FAMILY_DIR : /home/temporaryuser4/Desktop/rufiz/~/AnimalClassification/AnimalClassification_server/models/cnn_scratch/customcnn_v1
TRACKING_DIR     : /home/temporaryuser4/Desktop/rufiz/~/AnimalClassification/AnimalClassification_server/mlruns


In [ ]:
# Cell 3 - Add PROJECT_ROOT to sys.path and import project modules

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data.transforms import load_transforms_config, get_train_transforms, get_eval_transforms
from src.models.cnn_scratch.models import build_model
from src.models.cnn_scratch.utils import (
    atomic_save_json,
    benchmark_inference,
    build_metrics_payload,
    build_training_config,
    count_parameters,
    ensure_dir,
    evaluate_model,
    export_model_to_onnx,
    fit_model,
    make_run_dir,
    model_size_mb_from_state_dict,
    restore_best_weights,
    save_checkpoint_atomic,
    save_training_curves,
)

In [ ]:
# Cell 4 - Determinism helpers

SEED = 42

def seed_everything(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(SEED)

torch.backends.cudnn.deterministic = False
torch.backends.cudnn.benchmark = True

print("[OK] Seeds initialized:", SEED)

[OK] Seeds initialized: 42


In [ ]:
# Cell 5 - Required file checks

required_paths = {
    "train_csv": TRAIN_CSV,
    "val_csv": VAL_CSV,
    "test_csv": TEST_CSV,
    "classes_json": CLASSES_JSON,
    "transforms_yaml": TRANSFORMS_CONFIG_PATH,
}

missing_required = [name for name, path in required_paths.items() if not path.exists()]
if missing_required:
    details = "\n".join(f"- {name}: {required_paths[name]}" for name in missing_required)
    raise FileNotFoundError(f"Missing required files:\n{details}")

print("[OK] Required files exist.")

[OK] Required files exist.


In [ ]:
# Cell 6 - MLflow setup

mlflow.set_tracking_uri(TRACKING_DIR.as_uri())
mlflow.set_experiment("AnimalClassification")

print("MLflow tracking URI:", TRACKING_DIR.as_uri())

MLflow tracking URI: file:///home/temporaryuser4/Desktop/rufiz/~/AnimalClassification/AnimalClassification_server/mlruns


/home/temporaryuser4/Desktop/rufiz/~/AnimalClassification/AnimalClassification_server/.venv/lib/python3.12/site-packages/mlflow/tracking/_tracking_service/utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)


## Data Loading & Preprocessing

**Cells 7–11** load the dataset and prepare it for training:
- **Cell 7** reads `train/val/test.csv` and `classes.json` → **50,127 train / 6,266 val / 6,266 test** samples across 3 classes (`cats`, `dogs`, `wildlife`).
- **Cell 8** normalises filepaths across OS/machine differences and drops any missing files (none missing here).
- **Cell 9** loads `transforms_v1.yaml`: train pipeline adds random augmentations; eval pipeline resizes to 256 px, centre-crops to 224×224 and applies ImageNet normalisation.
- **Cell 10-11** defines `CSVDataset` (inline, Run-All safe) and wraps each split in a `DataLoader` (batch 64 on CUDA, 8 workers, pin memory).

In [ ]:
# Cell 7 - Load split manifests and class mapping

train_df = pd.read_csv(TRAIN_CSV)
val_df = pd.read_csv(VAL_CSV)
test_df = pd.read_csv(TEST_CSV)

with open(CLASSES_JSON, "r", encoding="utf-8") as f:
    class_to_idx = json.load(f)

idx_to_class = {v: k for k, v in class_to_idx.items()}
NUM_CLASSES = len(class_to_idx)

print("train:", len(train_df), "val:", len(val_df), "test:", len(test_df))
print("class_to_idx:", class_to_idx)

train: 50127 val: 6266 test: 6266
class_to_idx: {'cats': 0, 'dogs': 1, 'wildlife': 2}


In [ ]:
# Cell 8 - Normalize filepaths and filter missing files

def normalize_filepath(p: str) -> str:
    p = str(p).strip()
    p2 = p.replace("\\", "/")

    marker = "/data/prepared/"
    low = p2.lower()
    if marker in low:
        idx = low.find(marker)
        rel = p2[idx + 1:]
        return str((PROJECT_ROOT / rel).resolve())

    if p2.startswith("data/"):
        return str((PROJECT_ROOT / p2).resolve())

    return str(Path(p).expanduser().resolve()) if Path(p).exists() else p

for df in (train_df, val_df, test_df):
    df["filepath"] = df["filepath"].apply(normalize_filepath)
    df["label_idx"] = df["label"].map(class_to_idx)
    assert df["label_idx"].isna().sum() == 0

def filter_existing(df: pd.DataFrame, split_name: str) -> pd.DataFrame:
    exists = df["filepath"].apply(lambda p: Path(p).exists())
    missing_count = int((~exists).sum())
    if missing_count > 0:
        print(f"[WARNING] Dropping {missing_count} missing files from {split_name} split.")
    return df[exists].copy()

train_df = filter_existing(train_df, "train")
val_df = filter_existing(val_df, "val")
test_df = filter_existing(test_df, "test")

print("Filtered sizes:", len(train_df), len(val_df), len(test_df))

Filtered sizes: 50127 6266 6266


In [ ]:
# Cell 9 - Load transforms

tfm_cfg = load_transforms_config(str(TRANSFORMS_CONFIG_PATH))
train_tfm = get_train_transforms(tfm_cfg)
eval_tfm = get_eval_transforms(tfm_cfg)

print("[OK] Transforms loaded.")

[OK] Transforms loaded.


In [ ]:
# Cell 10 - Dataset definition (portable, notebook-local, Run-All safe)

from torch.utils.data import Dataset

class CSVDataset(Dataset):
    def __init__(self, df: pd.DataFrame, transform=None):
        self.df = df.reset_index(drop=True).copy()
        self.paths = self.df["filepath"].tolist()
        self.labels = self.df["label_idx"].astype(np.int64).tolist()
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx: int):
        from PIL import Image
        img = Image.open(self.paths[idx]).convert("RGB")
        if self.transform is not None:
            img = self.transform(img)
        y = self.labels[idx]
        return img, y

train_ds = CSVDataset(train_df, transform=train_tfm)
val_ds = CSVDataset(val_df, transform=eval_tfm)
test_ds = CSVDataset(test_df, transform=eval_tfm)

print("[OK] Datasets built.")

[OK] Datasets built.


In [ ]:
# Cell 11 - Device and DataLoaders

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
NUM_WORKERS = min(8, os.cpu_count() or 2)
PIN_MEMORY = True if DEVICE == "cuda" else False

BATCH_SIZE = 64 if DEVICE == "cuda" else 16

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
)

val_loader = DataLoader(
    val_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
)

test_loader = DataLoader(
    test_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
)

print("DEVICE      :", DEVICE)
print("BATCH_SIZE  :", BATCH_SIZE)
print("NUM_WORKERS :", NUM_WORKERS)
print("PIN_MEMORY  :", PIN_MEMORY)

DEVICE      : cuda
BATCH_SIZE  : 64
NUM_WORKERS : 8
PIN_MEMORY  : True


## Pre-Training Checks

**Cells 12–15** run final validation before the training loop begins:
- **Cell 12** fetches one batch and asserts shape `(64, 3, 224, 224)`, 1-D labels, and no `NaN`/`Inf` values.
- **Cell 13** defines all hyperparameters (Adam, lr=1e-3, weight_decay=1e-4, dropout=0.5, 30 epochs, ReduceLROnPlateau patience=3) and serialises them into `config_payload`.
- **Cell 14** creates a timestamped run directory (`run_YYYYMMDD_HHMMSS/`) and saves `config.json` immediately - so the config is preserved even if training crashes later.

In [ ]:
# Cell 12 - Batch sanity check

xb, yb = next(iter(train_loader))

assert xb.ndim == 4 and xb.shape[1:] == (3, 224, 224), f"Unexpected batch shape: {tuple(xb.shape)}"
assert yb.ndim == 1, f"Unexpected labels shape: {tuple(yb.shape)}"
assert torch.isfinite(xb).all(), "Non-finite values in training batch."

print("[OK] Batch sanity check passed:", tuple(xb.shape), tuple(yb.shape))

[OK] Batch sanity check passed: (64, 3, 224, 224) (64,)


In [ ]:
# Cell 13 - Hyperparameters and training config

EPOCHS = 30
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
DROPOUT_P = 0.5
GRAD_CLIP_MAX_NORM = 1.0

training_params = {
    "epochs": EPOCHS,
    "batch_size": BATCH_SIZE,
    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
    "dropout_p": DROPOUT_P,
    "grad_clip_max_norm": GRAD_CLIP_MAX_NORM,
    "optimizer": "Adam",
    "scheduler": {
        "name": "ReduceLROnPlateau",
        "mode": "min",
        "factor": 0.3,
        "patience": 3,
    },
    "loss_fn": "CrossEntropyLoss",
    "seed": SEED,
}

config_payload = build_training_config(
    model_name=MODEL_NAME,
    split_id=SPLIT_ID,
    transform_id_train=TRANSFORM_ID_TRAIN,
    transform_id_eval=TRANSFORM_ID_EVAL,
    dataset_version="prepared_dataset_current",
    training_params=training_params,
    extra={
        "num_classes": NUM_CLASSES,
        "class_to_idx": class_to_idx,
    },
)

print(json.dumps(config_payload, indent=2))

{
  "model_name": "customcnn_v1",
  "split_id": "split_v1",
  "transform_id_train": "transforms_v1_train_runtime_aug",
  "transform_id_eval": "transforms_v1_eval_resize256_centercrop224_imagenetnorm",
  "dataset_version": "prepared_dataset_current",
  "training_params": {
    "epochs": 30,
    "batch_size": 64,
    "learning_rate": 0.001,
    "weight_decay": 0.0001,
    "dropout_p": 0.5,
    "grad_clip_max_norm": 1.0,
    "optimizer": "Adam",
    "scheduler": {
      "name": "ReduceLROnPlateau",
      "mode": "min",
      "factor": 0.3,
      "patience": 3
    },
    "loss_fn": "CrossEntropyLoss",
    "seed": 42
  },
  "timestamp": "2026-03-13T09:58:56.263832",
  "num_classes": 3,
  "class_to_idx": {
    "cats": 0,
    "dogs": 1,
    "wildlife": 2
  }
}


In [ ]:
# Cell 14 - Create run directory and save config early

RUN_DIR = make_run_dir(MODEL_FAMILY_DIR, prefix="run")
CHECKPOINT_PATH = RUN_DIR / "checkpoint.pt"
ONNX_PATH = RUN_DIR / "exported.onnx"
CONFIG_PATH = RUN_DIR / "config.json"
METRICS_PATH = RUN_DIR / "metrics.json"

atomic_save_json(CONFIG_PATH, config_payload)

print("RUN_DIR        :", RUN_DIR)
print("CHECKPOINT_PATH:", CHECKPOINT_PATH)
print("ONNX_PATH      :", ONNX_PATH)
print("CONFIG_PATH    :", CONFIG_PATH)
print("METRICS_PATH   :", METRICS_PATH)

RUN_DIR        : /home/temporaryuser4/Desktop/rufiz/~/AnimalClassification/AnimalClassification_server/models/cnn_scratch/customcnn_v1/run_20260313_095856
CHECKPOINT_PATH: /home/temporaryuser4/Desktop/rufiz/~/AnimalClassification/AnimalClassification_server/models/cnn_scratch/customcnn_v1/run_20260313_095856/checkpoint.pt
ONNX_PATH      : /home/temporaryuser4/Desktop/rufiz/~/AnimalClassification/AnimalClassification_server/models/cnn_scratch/customcnn_v1/run_20260313_095856/exported.onnx
CONFIG_PATH    : /home/temporaryuser4/Desktop/rufiz/~/AnimalClassification/AnimalClassification_server/models/cnn_scratch/customcnn_v1/run_20260313_095856/config.json
METRICS_PATH   : /home/temporaryuser4/Desktop/rufiz/~/AnimalClassification/AnimalClassification_server/models/cnn_scratch/customcnn_v1/run_20260313_095856/metrics.json


## Model Architecture

Builds `CustomCNNv1` from scratch - a lightweight 3-block CNN serving as the Phase 3 baseline:

| Block | Layers |
|---|---|
| Feature extractor | Conv(3→32) → ReLU → MaxPool × 3 stages (32→64→128 ch) |
| Classifier head | AdaptiveAvgPool → Flatten → Linear(128→256) → ReLU → Dropout(0.5) → Linear(256→3) |

**127,043 parameters · ~0.485 MB.** Also sets up `CrossEntropyLoss`, Adam optimizer, and `ReduceLROnPlateau` scheduler (factor=0.3, patience=3).

In [ ]:
# Cell 15 - Build model, optimizer, scheduler, criterion

model = build_model(
    model_name=MODEL_NAME,
    num_classes=NUM_CLASSES,
    input_channels=3,
    dropout_p=DROPOUT_P,
).to(DEVICE)

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.3,
    patience=3,
)

param_count = count_parameters(model)
size_mb = model_size_mb_from_state_dict(model)

print(model)
print("Parameter count:", param_count)
print("Approx model size (MB):", round(size_mb, 3))

CustomCNNv1(
  (features): Sequential(
    (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU(inplace=True)
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): ReLU(inplace=True)
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (6): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (7): ReLU(inplace=True)
    (8): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (classifier): Sequential(
    (0): AdaptiveAvgPool2d(output_size=1)
    (1): Flatten(start_dim=1, end_dim=-1)
    (2): Linear(in_features=128, out_features=256, bias=True)
    (3): ReLU(inplace=True)
    (4): Dropout(p=0.5, inplace=False)
    (5): Linear(in_features=256, out_features=3, bias=True)
  )
)
Parameter count: 127043
Approx model size (MB): 0.485


## Training (30 Epochs)

`fit_model` runs the full loop with gradient clipping (`max_norm=1.0`), per-epoch val metrics, and best-model tracking by val macro F1.

**Key observations:**
- Steady improvement from ~56% accuracy (Epoch 1) to ~94% (Epoch 30).
- LR decayed `1e-3 → 3e-4` at **Epoch 22** (scheduler triggered), causing a notable accuracy jump in Epochs 23–28.
- **Best epoch: 28** - val macro F1 = **0.9487**, val accuracy = **94.75%**.

> ⚠️ `PIL TiffImagePlugin` truncated-file warnings appear throughout - non-fatal, caused by a small number of malformed TIFF images in the dataset.

In [ ]:
# Cell 16 - Train model

history, best_state = fit_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=criterion,
    optimizer=optimizer,
    scheduler=scheduler,
    device=DEVICE,
    num_classes=NUM_CLASSES,
    epochs=EPOCHS,
    grad_clip_max_norm=GRAD_CLIP_MAX_NORM,
)

print("[OK] Training complete.")
print("Best epoch:", best_state.get("epoch"))
print("Best val macro F1:", best_state.get("best_val_macro_f1"))

/home/temporaryuser4/Desktop/rufiz/~/AnimalClassification/AnimalClassification_server/.venv/lib/python3.12/site-packages/PIL/TiffImagePlugin.py:949: UserWarning: Truncated File Read
  warnings.warn(str(msg))


[Epoch 01/30] train_loss=0.8974 train_acc=0.5604 val_loss=0.8318 val_acc=0.6029 val_f1=0.5802 lr=0.001000


/home/temporaryuser4/Desktop/rufiz/~/AnimalClassification/AnimalClassification_server/.venv/lib/python3.12/site-packages/PIL/TiffImagePlugin.py:949: UserWarning: Truncated File Read
  warnings.warn(str(msg))


[Epoch 02/30] train_loss=0.6624 train_acc=0.7072 val_loss=0.5066 val_acc=0.8000 val_f1=0.8064 lr=0.001000


/home/temporaryuser4/Desktop/rufiz/~/AnimalClassification/AnimalClassification_server/.venv/lib/python3.12/site-packages/PIL/TiffImagePlugin.py:949: UserWarning: Truncated File Read
  warnings.warn(str(msg))


...

[Epoch 28/30] train_loss=0.1552 train_acc=0.9421 val_loss=0.1412 val_acc=0.9475 val_f1=0.9487 lr=0.000300


/home/temporaryuser4/Desktop/rufiz/~/AnimalClassification/AnimalClassification_server/.venv/lib/python3.12/site-packages/PIL/TiffImagePlugin.py:949: UserWarning: Truncated File Read
  warnings.warn(str(msg))


[Epoch 29/30] train_loss=0.1545 train_acc=0.9439 val_loss=0.1631 val_acc=0.9386 val_f1=0.9394 lr=0.000300


/home/temporaryuser4/Desktop/rufiz/~/AnimalClassification/AnimalClassification_server/.venv/lib/python3.12/site-packages/PIL/TiffImagePlugin.py:949: UserWarning: Truncated File Read
  warnings.warn(str(msg))


[Epoch 30/30] train_loss=0.1574 train_acc=0.9417 val_loss=0.1488 val_acc=0.9465 val_f1=0.9481 lr=0.000300
[OK] Training complete.
Best epoch: 28
Best val macro F1: 0.9487344677661979


## Post-Training: Checkpoint & Evaluation

**Cells 17–19** finalise the trained model:
- **Cell 17** restores best-epoch weights and saves `checkpoint.pt` atomically (model + optimizer state, best metrics, class map, config).
- **Cell 18** evaluates on the held-out test set → **loss 0.1442 · accuracy 94.54% · macro F1 0.9472** (closely matching val, confirming no overfitting).
- **Cell 19** benchmarks inference speed (5 warmup + 20 timed batches) → **~0.19 ms/image · ~5,153 img/sec** on CUDA.

In [ ]:
# Cell 17 - Restore best weights and save checkpoint

model = restore_best_weights(model, best_state)

checkpoint_payload = {
    "model_name": MODEL_NAME,
    "model_state_dict": model.state_dict(),
    "optimizer_state_dict": best_state.get("optimizer_state_dict"),
    "best_epoch": best_state.get("epoch"),
    "best_val_macro_f1": best_state.get("best_val_macro_f1"),
    "best_val_loss": best_state.get("best_val_loss"),
    "best_val_accuracy": best_state.get("best_val_accuracy"),
    "class_to_idx": class_to_idx,
    "config": config_payload,
}

save_checkpoint_atomic(CHECKPOINT_PATH, checkpoint_payload)

print("[OK] Checkpoint saved:", CHECKPOINT_PATH)

[OK] Checkpoint saved: /home/temporaryuser4/Desktop/rufiz/~/AnimalClassification/AnimalClassification_server/models/cnn_scratch/customcnn_v1/run_20260313_095856/checkpoint.pt


In [ ]:
# Cell 18 - Evaluate best model on test set

test_metrics = evaluate_model(
    model=model,
    loader=test_loader,
    criterion=criterion,
    device=DEVICE,
    num_classes=NUM_CLASSES,
)

print("Test loss     :", round(test_metrics["loss"], 4))
print("Test accuracy :", round(test_metrics["accuracy"], 4))
print("Test macro F1 :", round(test_metrics["macro_f1"], 4))

Test loss     : 0.1442
Test accuracy : 0.9454
Test macro F1 : 0.9472


In [ ]:
# Cell 19 - Benchmark inference cost on test loader

benchmark_metrics = benchmark_inference(
    model=model,
    loader=test_loader,
    device=DEVICE,
    warmup_batches=5,
    timed_batches=20,
)

print(json.dumps(benchmark_metrics, indent=2))

{
  "latency_ms_per_batch": 12.420012049551588,
  "latency_ms_per_image": 0.19406268827424356,
  "throughput_img_per_sec": 5152.9740667450205,
  "num_timed_batches": 20.0
}


## Artifact Saving & Experiment Logging

**Cells 20–24** persist all run artifacts and log them to MLflow:
- **Cell 20** saves `loss_curve.png` and `accuracy_curve.png` to the run directory.
- **Cell 21** attempts ONNX export (opset 17) - **failed** this run (`ModuleNotFoundError: onnxscript`). Fix: `pip install onnxscript`. Non-fatal; all other artifacts remain valid.
- **Cell 22** assembles and saves `metrics.json` (history, best state, test metrics, benchmark, ONNX status).
- **Cell 23** copies metrics to `reports/metrics/` for flat cross-run browsing.
- **Cell 24** logs all params, metrics, and artifact files to the `AnimalClassification` MLflow experiment.

In [ ]:
# Cell 20 - Save training curves

curve_paths = save_training_curves(history, RUN_DIR)

print("Saved curves:")
for name, path in curve_paths.items():
    print(f"  - {name}: {path}")

Saved curves:
  - loss_curve: /home/temporaryuser4/Desktop/rufiz/~/AnimalClassification/AnimalClassification_server/models/cnn_scratch/customcnn_v1/run_20260313_095856/loss_curve.png
  - accuracy_curve: /home/temporaryuser4/Desktop/rufiz/~/AnimalClassification/AnimalClassification_server/models/cnn_scratch/customcnn_v1/run_20260313_095856/accuracy_curve.png


In [ ]:
# Cell 21 - Export ONNX safely (non-fatal if export support is unavailable)

onnx_export_status = {
    "attempted": True,
    "succeeded": False,
    "path": str(ONNX_PATH),
    "error": None,
}

try:
    export_model_to_onnx(
        model=model,
        export_path=ONNX_PATH,
        input_shape=(1, 3, 224, 224),
        device=DEVICE,
        opset_version=17,
    )
    onnx_export_status["succeeded"] = True
    print("[OK] ONNX exported:", ONNX_PATH)

except Exception as e:
    onnx_export_status["error"] = f"{type(e).__name__}: {e}"
    print("[WARNING] ONNX export failed. Training artifacts remain valid.")
    print("ONNX export error:", onnx_export_status["error"])

[WARNING] ONNX export failed. Training artifacts remain valid.
ONNX export error: ModuleNotFoundError: No module named 'onnxscript'


In [ ]:
# Cell 22 - Build and save metrics payload

metrics_payload = build_metrics_payload(
    history=history,
    best_state=best_state,
    test_metrics=test_metrics,
    benchmark_metrics=benchmark_metrics,
    parameter_count=param_count,
    model_size_mb=size_mb,
)

metrics_payload["onnx_export"] = onnx_export_status

atomic_save_json(METRICS_PATH, metrics_payload)

print("[OK] Metrics saved:", METRICS_PATH)

[OK] Metrics saved: /home/temporaryuser4/Desktop/rufiz/~/AnimalClassification/AnimalClassification_server/models/cnn_scratch/customcnn_v1/run_20260313_095856/metrics.json


In [ ]:
# Cell 23 - Save lightweight report copies to reports/metrics

report_metrics_path = REPORTS_METRICS_DIR / f"{MODEL_NAME}_{RUN_DIR.name}_metrics.json"
atomic_save_json(report_metrics_path, metrics_payload)

print("[OK] Report metrics copy saved:", report_metrics_path)

[OK] Report metrics copy saved: /home/temporaryuser4/Desktop/rufiz/~/AnimalClassification/AnimalClassification_server/reports/metrics/customcnn_v1_run_20260313_095856_metrics.json


In [ ]:
# Cell 24 - MLflow logging

with mlflow.start_run(run_name=f"{MODEL_NAME}_{RUN_DIR.name}"):
    mlflow.log_param("stage", "cnn_scratch_training")
    mlflow.log_param("model_name", MODEL_NAME)
    mlflow.log_param("split_id", SPLIT_ID)
    mlflow.log_param("transform_id_train", TRANSFORM_ID_TRAIN)
    mlflow.log_param("transform_id_eval", TRANSFORM_ID_EVAL)
    mlflow.log_param("device", DEVICE)
    mlflow.log_param("epochs", EPOCHS)
    mlflow.log_param("batch_size", BATCH_SIZE)
    mlflow.log_param("learning_rate", LEARNING_RATE)
    mlflow.log_param("weight_decay", WEIGHT_DECAY)
    mlflow.log_param("dropout_p", DROPOUT_P)
    mlflow.log_param("grad_clip_max_norm", GRAD_CLIP_MAX_NORM)
    mlflow.log_param("num_workers", NUM_WORKERS)
    mlflow.log_param("parameter_count", param_count)
    mlflow.log_param("model_size_mb", size_mb)
    mlflow.log_param("onnx_export_succeeded", onnx_export_status["succeeded"])

    mlflow.log_metric("best_epoch", float(best_state.get("epoch", -1)))
    mlflow.log_metric("best_val_macro_f1", float(best_state.get("best_val_macro_f1", float("nan"))))
    mlflow.log_metric("best_val_loss", float(best_state.get("best_val_loss", float("nan"))))
    mlflow.log_metric("best_val_accuracy", float(best_state.get("best_val_accuracy", float("nan"))))

    mlflow.log_metric("test_loss", float(test_metrics["loss"]))
    mlflow.log_metric("test_accuracy", float(test_metrics["accuracy"]))
    mlflow.log_metric("test_macro_f1", float(test_metrics["macro_f1"]))

    mlflow.log_metric("latency_ms_per_batch", float(benchmark_metrics["latency_ms_per_batch"]))
    mlflow.log_metric("latency_ms_per_image", float(benchmark_metrics["latency_ms_per_image"]))
    mlflow.log_metric("throughput_img_per_sec", float(benchmark_metrics["throughput_img_per_sec"]))

    mlflow.log_artifact(str(CONFIG_PATH))
    mlflow.log_artifact(str(METRICS_PATH))
    mlflow.log_artifact(str(CHECKPOINT_PATH))
    mlflow.log_artifact(str(curve_paths["loss_curve"]))
    mlflow.log_artifact(str(curve_paths["accuracy_curve"]))

    if onnx_export_status["succeeded"] and ONNX_PATH.exists():
        mlflow.log_artifact(str(ONNX_PATH))

print("[OK] MLflow logging complete.")

[OK] MLflow logging complete.


## Result

This notebook completed the full Phase 3 training run for `CustomCNN v1` and produced:

- checkpoint
- run config
- metrics
- training curves
- MLflow logs
- inference benchmarking metrics

The next notebook should reuse the same structure for:

- `30_02_customcnn_v2.ipynb`

Only the model name and architecture-specific settings should change.